In [1]:
# ── Cell 1 — Imports ───────────────────────────────────────────────────────
import requests, time, json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

Path('../data/idigbio').mkdir(parents=True, exist_ok=True)

TAXA = [
    'Mammuthus', 'Mammut', 'Equus', 'Camelops',
    'Paramylodon', 'Bison', 'Arctodus', 'Cervus'
]
STATES = ['Washington', 'Oregon', 'Nevada']
RECORDS = []

In [3]:
# ── Cell 2 — Harvest iDigBio (fixed API format) ────────────────────────────
BASE = 'https://search.idigbio.org/v2/search/records'

for taxon in TAXA:
    for state in STATES:
        payload = {
            'rq': {
                'genus': taxon.lower(),
                'stateprovince': state.lower()
            },
            'limit': 100,
            'offset': 0
        }
        try:
            r = requests.post(BASE,
                              json=payload,
                              headers={'Content-Type': 'application/json'},
                              timeout=60)
            if r.status_code == 200:
                data = r.json()
                hits = data.get('items', [])
                found = 0
                for h in hits:
                    rec = h.get('data', {})
                    idx = h.get('indexTerms', {})
                    gp  = idx.get('geopoint', {})
                    lat = gp.get('lat') if gp else None
                    lon = gp.get('lon') if gp else None
                    if lat and lon:
                        RECORDS.append({
                            'taxon':     taxon,
                            'genus':     rec.get('dwc:genus', taxon),
                            'species':   rec.get('dwc:specificEpithet', ''),
                            'state':     state,
                            'county':    rec.get('dwc:county', ''),
                            'latitude':  float(lat),
                            'longitude': float(lon),
                            'source':    'idigbio',
                            'inst':      rec.get('dwc:institutionCode', ''),
                        })
                        found += 1
                print(f'  {taxon} / {state}: {len(hits)} hits, {found} with coords')
            else:
                print(f'  {taxon} / {state}: HTTP {r.status_code} — {r.text[:100]}')
        except Exception as e:
            print(f'  {taxon} / {state}: ERROR {e}')
        time.sleep(0.5)

df = pd.DataFrame(RECORDS) if RECORDS else pd.DataFrame()
print(f'\nTotal iDigBio records with coordinates: {len(df)}')
if not df.empty:
    print(df.groupby(['taxon','state']).size().unstack(fill_value=0).to_string())
else:
    print('No records found — check API response above')

  Mammuthus / Washington: 25 hits, 16 with coords
  Mammuthus / Oregon: 68 hits, 11 with coords
  Mammuthus / Nevada: 30 hits, 0 with coords
  Mammut / Washington: 3 hits, 0 with coords
  Mammut / Oregon: 100 hits, 74 with coords
  Mammut / Nevada: 22 hits, 0 with coords
  Equus / Washington: 47 hits, 7 with coords
  Equus / Oregon: 100 hits, 0 with coords
  Equus / Nevada: 100 hits, 3 with coords
  Camelops / Washington: 0 hits, 0 with coords
  Camelops / Oregon: 100 hits, 0 with coords
  Camelops / Nevada: 100 hits, 0 with coords
  Paramylodon / Washington: 2 hits, 2 with coords
  Paramylodon / Oregon: 30 hits, 7 with coords
  Paramylodon / Nevada: 0 hits, 0 with coords
  Bison / Washington: 37 hits, 30 with coords
  Bison / Oregon: 100 hits, 57 with coords
  Bison / Nevada: 33 hits, 1 with coords
  Arctodus / Washington: 0 hits, 0 with coords
  Arctodus / Oregon: 3 hits, 1 with coords
  Arctodus / Nevada: 1 hits, 1 with coords
  Cervus / Washington: 100 hits, 89 with coords
  Cervus

In [4]:
# ── Cell 3 — Save ──────────────────────────────────────────────────────────
df.to_csv('../data/idigbio/idigbio_occurrences.csv', index=False)
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(r.longitude, r.latitude) for _, r in df.iterrows()],
    crs='EPSG:4326'
)
gdf.to_file('../data/idigbio/idigbio_occurrences.geojson', driver='GeoJSON')
print(f'Saved {len(df)} iDigBio records')

Saved 304 iDigBio records
